# Evaluation and Reliability Deep Dive

How to test, gate, and trust systems built on a non-deterministic component.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice. It goes deeper than notebook 19 and is
interview-focused: read the concept, run the example, then complete the
exercises.

## Learning Objectives

- Explain *why* LLMs are non-deterministic even at temperature 0, and why that breaks traditional testing.
- Build an eval suite: unit-style evals, end-to-end evals, offline vs online, CI regression gating.
- Use LLM-as-judge correctly: rubrics, pairwise vs pointwise, known biases, calibration against humans.
- Build, version, and maintain a golden dataset sourced from production traces.
- Describe the 2026 observability stack (tracing, cost tracking, the data flywheel).

## Mental Model

Treat the LLM as a **stochastic component wrapped in deterministic scaffolding**.
You cannot assert exact outputs, so you assert *properties* of outputs, score
*distributions* of behavior, and gate changes on *aggregate metrics* over a
versioned dataset — exactly how you'd treat a noisy sensor in classical
engineering.

The reliability loop:

```
production traces + user feedback
        │  (filter, annotate)
        ▼
golden dataset (versioned)
        │  (offline evals in CI)
        ▼
prompt/model change gated on metric delta
        │  (deploy)
        ▼
online evals on sampled traffic ──▶ back to traces
```

## Important Concepts

- **Non-determinism**: sampling randomness, floating-point non-associativity, batch non-invariance, MoE expert routing.
- **Eval taxonomy**: unit-style (single call, per-PR), end-to-end/agent (full trace, tool calls, trajectory), offline (golden set, pre-deploy), online (sampled prod traffic, judges, drift detection).
- **Eval-driven development (EDD)**: write the eval before the prompt — TDD for LLMs.
- **LLM-as-judge**: a (stronger, different-family) model scoring outputs against a rubric.
- **Golden dataset**: versioned, curated examples with expected properties; the regression suite.
- **Observability**: hierarchical traces (LLM/tool/retrieval spans), token+cost accounting, feedback attached to spans.

## Need-To-Know Coverage Checklist

- [x] Why temperature 0 is still non-deterministic (batch non-invariance is the key insight).
- [x] Property-based assertions, semantic-similarity thresholds, multiple-run consistency (pass@k).
- [x] Offline vs online evals; regression gating in CI with tolerance bands.
- [x] LLM-as-judge biases: position, self-preference, verbosity, sycophancy — and mitigations.
- [x] Judge calibration against human labels (Cohen's kappa), judge prompt versioning.
- [x] Golden dataset sourcing (four buckets), size guidance, drift handling.
- [x] Tooling landscape: Braintrust, LangSmith, Promptfoo, DeepEval, Langfuse, Helicone, Arize Phoenix.
- [x] Guardrails: input/output rails, PII (Presidio), confidence-threshold abstention, guardrail hit rates as drift signals.
- [x] OpenTelemetry GenAI semantic conventions (`gen_ai.*` spans).

## Deep Study Notes

### Why non-determinism exists (the interview-grade answer)

Most candidates say "temperature." The deeper answer:

1. **Batch non-invariance (primary culprit).** Thinking Machines Lab's
   "Defeating Nondeterminism in LLM Inference" (2025) showed the real cause:
   GPU reduction kernels (RMSNorm, matmul, attention) tile differently
   depending on dynamic batch size. Your request is co-batched with strangers'
   requests, so floating-point summation *order* changes per request.
2. **Floating-point non-associativity.** (a+b)+c != a+(b+c) in bf16. A
   last-bit logit difference occasionally flips the argmax token; because
   generation is autoregressive, one flipped token cascades into a completely
   different completion.
3. **MoE routing.** In mixture-of-experts models, expert-capacity limits mean
   batch composition can change *which experts* process your tokens — a
   structural (not just numerical) source of variance.
4. Fixes exist: batch-invariant kernels are now in vLLM and SGLang, giving
   bit-identical outputs. Hosted APIs offer `seed` + `system_fingerprint` as
   best-effort only.

### Why this breaks testing, and what to do instead

`assert output == expected` is dead: same input, different bytes. Replace with:

- **Property-based assertions** — assert invariants, not strings: "output
  contains no PII", "cited doc ids exist in the retrieval set", "JSON parses
  and totals sum correctly".
- **Semantic similarity thresholds** — embedding cosine similarity (or a judge)
  against a reference, with a tuned threshold. Exact match shows >30%
  false-fail rates on correct-but-paraphrased answers.
- **Multiple-run consistency** — run N times; gate on "passes >= k of N"
  (pass@k / pass^k semantics). In CI, use tolerance bands on aggregate scores,
  never exact thresholds, or the build flakes.

### Building an eval suite

- **Unit-style evals**: single LLM call, assertion or judge score, run per-PR.
- **End-to-end/agent evals**: score whole traces — right tool called with right
  args, sensible trajectory, correct final answer over multi-turn runs.
- **Offline**: golden dataset run pre-deploy; catches known failure modes.
- **Online**: sample 5–10% of production traffic, score with automated judges,
  watch for drift and *novel* failures a static set can't catch.
- **CI gating**: pin the judge model, use a stable golden subset, tolerance
  bands; fail the build on regression.
- **EDD**: write evals before prompts. "Evals engineer" is a real job title now.

Tooling in one line each: **Braintrust** (full lifecycle, enterprise leader),
**LangSmith** (deepest on LangChain/LangGraph), **Promptfoo** (YAML/CLI CI
gating + red-teaming; reportedly acquired by OpenAI in 2026, core still
open-source), **DeepEval** (pytest-native, 50+ metrics — the standard per-PR
gate), **Langfuse** (MIT, self-hostable tracing+evals), **Helicone**
(proxy-based — one endpoint change buys caching and cost/latency dashboards;
fastest to deploy), **Arize Phoenix** (OTel-native, embedding drift).
Common pattern: a lightweight CI gate (DeepEval/Promptfoo) plus a platform for
dashboards and annotation (Braintrust/LangSmith/Langfuse).

### Guardrails (where they sit relative to evals)

Evals measure quality after the fact; **guardrails enforce constraints at
request time**, on both sides of the model:

- **Input rails**: prompt-injection/jailbreak detection (classifier models),
  topic control, PII scrubbing before the prompt (Microsoft **Presidio** is
  the default PII engine).
- **Output rails**: schema validation (now largely native in provider APIs),
  moderation/toxicity filters (Llama Guard, OpenAI Moderation), grounding
  checks, and **confidence thresholds** — below a threshold, abstain or route
  to a human instead of answering.
- Frameworks: **Guardrails AI** (validator hub, re-ask on failure) and
  **NVIDIA NeMo Guardrails** (programmable dialogue rails). Gateways (LiteLLM)
  expose guardrail hooks so enforcement happens at one choke point.
- The connection to this notebook: guardrail *hit rates* are themselves
  metrics — a spike in schema-validation failures or injection detections is
  an early drift signal that should page someone and feed the golden dataset.

### LLM-as-judge, done right

- **Rubrics beat holistic scores**: decompose into criteria, score each
  separately; binary pass/fail per criterion is more reliable than 1–10 scales.
- **Pairwise vs pointwise**: pairwise (A vs B) is more discriminative for model
  comparison and ranking; pointwise with rubrics scales for production
  monitoring and gating.
- **Known biases**: *position bias* (favors first answer — evaluate both
  orderings and require agreement), *self-preference* (a judge scores its own
  model family 10–25% higher — use a different, stronger family or a panel),
  *verbosity bias*, *sycophancy toward confident tone*.
- **Calibration**: build a human-labeled set (a few hundred examples), measure
  judge–human agreement (Cohen's kappa), iterate the judge prompt until
  agreement is acceptable, then **version the judge prompt like production
  code** and re-calibrate whenever the judge model changes.

### Golden datasets

- **Four-bucket framework**: (1) stratified sample of production traffic,
  (2) adversarial library, (3) hand-constructed edge cases, (4) replays of
  every shipped failure (each incident becomes a permanent regression case).
- **Size**: 20–50 cases gives signal; 100–300 is a dependable gate; 500–1000+
  for mature systems. Coverage stratification beats raw size.
- **Drift**: golden sets rot — add real failures weekly, re-review labels on
  policy changes, prune obsolete cases, and pair with online evals (static
  sets miss novel failures).

### Observability

- **OpenTelemetry GenAI semantic conventions** standardized tracing: the
  `gen_ai.*` attribute namespace (model, token usage, tool calls; agent/
  workflow/tool/model span types). Instrument once, route to any backend.
- Expected capabilities: hierarchical spans per LLM/tool/retrieval step,
  token+cost accounting per trace/user/feature, latency percentiles, user
  feedback attached to spans.
- **The flywheel** is the point: prod traces + feedback → annotated → appended
  to golden datasets → offline evals → better prompts → redeploy.

## Common Failure Modes

- Exact-match assertions on free text → flaky CI, then people ignore CI.
- Judge model unpinned → eval scores shift when the provider silently updates.
- Judge from the same model family as the system under test → inflated scores.
- Golden set never updated → evals pass while production quality degrades.
- Only offline evals → novel production failure modes invisible for weeks.
- Scoring 1–10 holistically → low inter-run agreement, meaningless deltas.
- Shipping a model swap because "it looks better in a few chats" instead of an eval delta.

## Implementation Notes

- Start with 20 golden cases and property assertions; add a judge only when
  properties can't express quality.
- Keep eval definitions in the repo, next to prompts, reviewed in PRs.
- Log every production trace with trace/span ids from day one — retrofitting
  observability is far more expensive than adding it early.
- Budget for evals: a judge call per output per metric adds up; sample online
  traffic rather than scoring 100%.

In [ ]:
"""Minimal eval harness: property-based checks + multi-run consistency.

No API needed — we simulate a non-deterministic model to show the pattern.
Swap `fake_model` for a real client in practice.
"""
import json
import random


def fake_model(prompt: str, seed: int) -> str:
    """Simulates non-determinism: same prompt, varying output."""
    rng = random.Random(seed)
    themes = ["onboarding friction", "pricing confusion"]
    if rng.random() < 0.15:  # occasional failure: invalid JSON
        return "The main themes are onboarding and pricing."
    picked = themes[: rng.randint(1, 2)]
    return json.dumps({"themes": picked, "n_respondents": 12})


def check_properties(raw: str) -> list[str]:
    """Property-based assertions: invariants, not exact strings."""
    failures = []
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        return ["output is not valid JSON"]
    if not isinstance(data.get("themes"), list) or not data["themes"]:
        failures.append("themes must be a non-empty list")
    if not isinstance(data.get("n_respondents"), int) or data["n_respondents"] < 0:
        failures.append("n_respondents must be a non-negative int")
    return failures


def run_eval(prompt: str, runs: int = 10, pass_threshold: float = 0.8) -> dict:
    """Multiple-run consistency: gate on pass rate, not a single run."""
    results = [check_properties(fake_model(prompt, seed=i)) for i in range(runs)]
    passes = sum(1 for f in results if not f)
    pass_rate = passes / runs
    return {
        "pass_rate": pass_rate,
        "gate": "PASS" if pass_rate >= pass_threshold else "FAIL",
        "sample_failures": [f for f in results if f][:3],
    }


report = run_eval("Extract themes from these interview transcripts as JSON.")
print(json.dumps(report, indent=2))


## Practice

1. Add a *semantic* check to `check_properties`: themes must come from an
   allowed taxonomy list (simulating classify-into-taxonomy).
2. Implement a toy LLM-as-judge: a function that scores an answer 0/1 against
   a two-criterion rubric, and demonstrate position bias by swapping A/B order.
3. Sketch the four buckets of a golden dataset for an AI interview product:
   list 3 concrete cases per bucket.
4. Explain to a colleague why temperature 0 does not give reproducible
   outputs on a hosted API — without using the word "random".

## Design Checklist

- [ ] Every assertion is a property or thresholded score, never exact match.
- [ ] CI gate uses tolerance bands and a pinned judge model.
- [ ] Golden dataset is versioned, stratified, and fed by production failures.
- [ ] Judge calibrated against human labels; judge prompt is versioned.
- [ ] Online evals sample production traffic; drift alarms exist.
- [ ] Traces carry `gen_ai.*` attributes: model, tokens, cost, feedback.
- [ ] No prompt or model change ships without an eval delta.